In [ ]:
# note
 # dont run the complte file directly because it contain a part
 # where we find phylogeny distance of a species to other species which will be talking lost of time and computation

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
BASE = Path("../data/raw/Avonet/")

# Show all columns and wide output
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1200)
pd.set_option("display.max_colwidth", 100)

# Suppress mixed-type warnings from messy raw data
import warnings
warnings.filterwarnings("ignore")

In [ ]:
from scipy.stats import ks_2samp
from sklearn.metrics import mean_absolute_error, mean_squared_error
plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.family'] = 'sans-serif'
sns.set_style('whitegrid')

In [ ]:

avonet = pd.read_csv( BASE/ "avonet_uncleaned_final_dataset.csv")
avonet_crosswalk = pd.read_csv(BASE / "avonet_crosswalk.csv")



In [ ]:
avonet.shape

In [ ]:
avonet

In [ ]:
avonet.isnull().sum()

In [ ]:
avonet.columns

In [ ]:
df = avonet.replace(["MISSING", "nan", ""], pd.NA)

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100)
})

# sort (optional but useful)
missing_summary = missing_summary.sort_values(by="missing_percent", ascending=False)

print(missing_summary)

In [ ]:
quantitative_feature = ['beak_culmen_avg', 'beak_culmen_avg_m', 'beak_culmen_avg_f', 'beak_nares_avg',
                       'beak_nares_avg_m', 'beak_nares_avg_f', 'beak_width_avg', 'beak_width_avg_m', 'beak_width_avg_f',
                       'beak_depth_avg', 'beak_depth_avg_m', 'beak_depth_avg_f', 'tarsus_avg', 'tarsus_avg_m', 'tarsus_avg_f',
                       'wing_len_avg', 'wing_len_avg_m', 'wing_len_avg_f', 'kipps_avg', 'kipps_avg_m', 'kipps_avg_f', 'secondary_avg',
                       'secondary_avg_m', 'secondary_avg_f', 'hwi_avg', 'hwi_avg_m', 'hwi_avg_f', 'tail_avg', 'tail_avg_m', 'tail_avg_f',
                         'mass_avg', 
                        'lat_min', 'lat_max', 'lat_centroid', 'lon_centroid', 'range_size']

catagorical_feature = [  'habitat', 'habitat_density', 'migration', 'trophic_level', 
                       'trophic_niche', 'lifestyle']

## before cleaning


In [ ]:
df_before = df.copy()

## Replace with Median & Mode

- median = quantitative feature
- mod = catagorical feature


In [ ]:
df1 = df.copy()

In [ ]:
for col in quantitative_feature:
    if df1[col].isnull().any():
        df1[col] = df1[col].fillna(df1[col].median())

In [ ]:
for col in catagorical_feature:
    df1[col] = df1[col].fillna(df1[col].mode()[0])

## Replace with Mean & Mode

- mean = quantitative feature
- mod = catagorical feature


In [ ]:
df2 = df.copy()

In [ ]:
for col in quantitative_feature:
    if df2[col].isnull().any():
        df2[col].fillna(df2[col].mean(), inplace=True)

In [ ]:
for col in catagorical_feature:
    df2[col] = df2[col].fillna(df2[col].mode()[0])

## Mean family/order wise and mod family/order wise


In [ ]:
df3 = df.copy()

In [ ]:
cols = ["avibase_id","family_birdlife", "order_birdlife","family_birdtree","order_birdtree"]

df3 = df3.merge(
    avonet_crosswalk[cols].drop_duplicates("avibase_id"),
    on="avibase_id",
    how="left"
)

In [ ]:
df3 = df3[cols + [c for c in df3.columns if c not in cols]]

In [ ]:
for col in quantitative_feature:
    df3[col] = df3[col].fillna(
        df3.groupby("family_birdlife")[col].transform("mean")
    )
    df3[col] = df3[col].fillna(
        df3.groupby("family_birdtree")[col].transform("mean")
    )
    # almost 99% covered in above 2 steps
    df3[col] = df3[col].fillna( df3[col].mean() )

In [ ]:
df3[catagorical_feature].isnull().sum()

In [ ]:
for col in catagorical_feature:
    df3[col] = df3[col].fillna(
        df3.groupby("family_birdlife")[col].transform(
            lambda x: x.mode().iloc[0] if not x.mode().empty else None
        )
    )
    
    df3[col] = df3[col].fillna(
        df3.groupby("family_birdtree")[col].transform(
            lambda x: x.mode().iloc[0] if not x.mode().empty else None
        )
    )
    
    # almost 99% covered in above 2 steps
    df3[col] = df3[col].fillna(df3[col].mode().iloc[0])

In [ ]:
df3[catagorical_feature].isnull().sum()

# Phylogeny based

- because compluting it takes so much time -> i did it once and store "avonet_cleaned_02.csv" , and commant out all code , and just this csv file futher


In [ ]:
from Bio import Phylo


tree = Phylo.read("../data/processed/Phylogeny.tre", "newick")

tree_species = [term.name for term in tree.get_terminals()]

print(len(tree_species))


In [ ]:
avonet_crosswalk["species_birdtree"] = (
    avonet_crosswalk["species_birdtree"]
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
    .str.capitalize()   # fixes Genus
)

In [ ]:
set_tree = set(tree_species)  # from .tre file
set_data = set(avonet_crosswalk["species_birdtree"].dropna())

# intersection → species present in BOTH
matched = set_data & set_tree

# results
print("Matched:", len(matched))
print("Not matched:", len(set_data - set_tree))
print("Total dataset species:", len(set_data))

In [ ]:
def precompute_depths(tree):
    depths = {}
    def walk(clade, dist):
        dist += (clade.branch_length or 0.0)
        if clade.is_terminal():
            depths[clade.name] = dist
        for child in clade.clades:
            walk(child, dist)
    walk(tree.root, 0.0)
    return depths

In [ ]:
depths = precompute_depths(tree)  # run once

In [ ]:
def get_closest_species(species_name: str, n: int = 25) -> dict:
    path = tree.get_path(species_name)
    target_depth = depths[species_name]
    collected = {}
    cumulative_up = 0.0

    for clade in reversed(path):
        cumulative_up += (clade.branch_length or 0.0)
        for term in clade.get_terminals():
            if term.name != species_name and term.name not in collected:
                collected[term.name] = cumulative_up + (depths[term.name] - (target_depth - cumulative_up))
        if len(collected) >= n:
            break

    return dict(sorted(collected.items(), key=lambda x: x[1])[:n])

In [ ]:
null_values = ["null", "NULL", "MISSING", "missing", "Missing", "NA", "na", "N/A", "n/a", ""]

def fill_null_quant(data, col):
    
    # normalize all null-like values to NaN
    data[col] = data[col].replace(null_values, np.nan)
    
    # loop only over rows where the target column is null
    for idx, row in data[data[col].isnull()].iterrows():
        
        # get the species name for this row
        species = row['species_birdtree']
        
        # if species name itself is null or not in the phylogenetic tree, skip
        if pd.isna(species) or species not in tree_species:
            continue
        
        # get 25 closest species with their distances {name: dist}
        neighbors = get_closest_species(species, n=25)
        
        # from those 25, keep only species that:
        # 1. exist in our dataset's species_birdtree column
        # 2. have a non-null value in the target column
        available = {sp: dist for sp, dist in neighbors.items()
                     if sp in data['species_birdtree'].values and 
                     pd.notna(data.loc[data['species_birdtree'] == sp, col].values[0])}
        
        # if none of the 25 neighbors are in dataset, skip
        if not available:
            continue
        
        # inverse distance weight — closer species get higher weight
        # +1e-9 to avoid division by zero when distance is 0
        weights = {sp: 1/(dist + 1e-9) for sp, dist in available.items()}
        
        # sum of all weights, used for normalization
        total = sum(weights.values())
        
        # normalize so all weights sum to 1
        norm_weights = {sp: w/total for sp, w in weights.items()}
        
        # weighted average — multiply each neighbor's value by its normalized weight and sum
        data.loc[idx, col] = sum(
            norm_weights[sp] * data.loc[data['species_birdtree'] == sp, col].values[0]
            for sp in available
        )
    
    return data

In [ ]:
from collections import defaultdict

def fill_null_cat(data, col):
    
    # normalize all null-like values to NaN
    data[col] = data[col].replace(null_values, np.nan)
    
    # loop only over rows where the target column is null
    for idx, row in data[data[col].isnull()].iterrows():
        
        # get the species name for this row
        species = row['species_birdtree']
        
        # if species name itself is null or not in the phylogenetic tree, skip
        if pd.isna(species) or species not in tree_species:
            continue
        
        # get 25 closest species with their distances {name: dist}
        neighbors = get_closest_species(species, n=25)
        
        # from those 25, keep only species that:
        # 1. exist in our dataset's species_birdtree column
        # 2. have a non-null value in the target column
        available = {sp: dist for sp, dist in neighbors.items()
                     if sp in data['species_birdtree'].values and 
                     pd.notna(data.loc[data['species_birdtree'] == sp, col].values[0])}
        
        # if none of the 25 neighbors are in dataset, skip
        if not available:
            continue
        
        # inverse distance weight — closer species get higher weight
        # +1e-9 to avoid division by zero when distance is 0
        weights = {sp: 1/(dist + 1e-9) for sp, dist in available.items()}
        
        # sum of all weights, used for normalization
        total = sum(weights.values())
        
        # normalize so all weights sum to 1
        norm_weights = {sp: w/total for sp, w in weights.items()}
        
        # sum weights per category instead of weighted average
        category_weights = defaultdict(float)
        for sp in available:
            category = data.loc[data['species_birdtree'] == sp, col].values[0]
            category_weights[category] += norm_weights[sp]
        
        # pick category with highest total weight
        data.loc[idx, col] = max(category_weights, key=category_weights.get)
    
    return data

In [ ]:
result = get_closest_species("Corvus_corax")
for species, dist in result.items():
    print(f"{species}: {dist:.4f}")

In [ ]:
df4 = df.copy()

In [ ]:
cols = ["avibase_id","species_birdtree","family_birdlife", "order_birdlife","family_birdtree","order_birdtree"]

df4 = df4.merge(
    avonet_crosswalk[cols].drop_duplicates("avibase_id"),
    on="avibase_id",
    how="left"
)

In [ ]:
df4 = df4[cols + [c for c in df3.columns if c not in cols]]

In [ ]:
null_values = ["null", "NULL", "MISSING", "missing", "Missing", "NA", "na", "N/A", "n/a", ""]

df4.replace(null_values, np.nan, inplace=True)

In [ ]:
# this below task takes very much time , thats why i done it ones and save it as file and use that file further 

In [ ]:
# for col in quantitative_feature:
#     df4 = fill_null_quant(df4, col)

In [ ]:
# for col in catagorical_feature:
#     df4 = fill_null_cat(df4, col)

In [ ]:
# df4.to_csv('avonet_cleaned_02.csv', index=False)

In [ ]:
data_phylogeny = pd.read_csv("../data/processed/avonet_cleaned.csv")

# comparing diferent cleaning strategies


In [ ]:
df_original = df_before.copy()
df_s1       = df1.copy()        
df_s2       = df3.copy()                  
df_s3       = data_phylogeny.copy() 

id_cols = ['avibase_id', 'species_birdtree', 'family_birdlife', 'order_birdlife', 'family_birdtree', 'order_birdtree',
           'total_individuals','female_count','male_count']

In [ ]:
analysis_cols    = [c for c in df_original.columns if c not in id_cols]
numerical_cols   = df_original[analysis_cols].select_dtypes(include=['number']).columns.tolist()
categorical_cols = df_original[analysis_cols].select_dtypes(include=['object', 'category']).columns.tolist()

print(f'Numerical   ({len(numerical_cols)})  : {numerical_cols}')
print(f'Categorical ({len(categorical_cols)}) : {categorical_cols}')

---

## Part 1 — Masked Validation (RMSE & MAE)

We take columns that have known values, hide 20% of them, impute using each strategy's column-level logic, and compare against the truth.


In [ ]:
np.random.seed(42)
colors  = {'S1': '#e07b54', 'S2': '#5b9bd5', 'S3': '#70ad47'}
results = []

for col in numerical_cols:
    known_idx = df_original[col].dropna().index.tolist()
    if len(known_idx) < 20:
        continue
    hide_idx  = np.random.choice(known_idx, size=int(len(known_idx) * 0.20), replace=False)
    truth     = df_original.loc[hide_idx, col].values
    for name, df in [('S1', df_s1), ('S2', df_s2), ('S3', df_s3)]:
        predicted = df.loc[hide_idx, col].values
        mask      = ~np.isnan(predicted)
        if mask.sum() == 0:
            continue
        results.append({
            'Column': col, 'Strategy': name,
            'MAE':  round(mean_absolute_error(truth[mask], predicted[mask]), 4),
            'RMSE': round(mean_squared_error(truth[mask], predicted[mask]) ** 0.5, 4)
        })

masked_df = pd.DataFrame(results)
masked_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric in zip(axes, ['MAE', 'RMSE']):
    pivot = masked_df.pivot(index='Column', columns='Strategy', values=metric)
    pivot.plot(kind='bar', ax=ax, color=[colors[c] for c in pivot.columns], edgecolor='white', width=0.7)
    ax.set_title(f'Masked Validation — {metric}', fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel(metric)
    ax.legend(title='Strategy')
    ax.tick_params(axis='x', rotation=45)
plt.suptitle('Lower is Better', fontsize=11, color='grey', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
summary_masked = masked_df.groupby('Strategy')[['MAE', 'RMSE']].mean().round(4)
summary_masked['Rank (MAE)']  = summary_masked['MAE'].rank().astype(int)
summary_masked['Rank (RMSE)'] = summary_masked['RMSE'].rank().astype(int)
summary_masked

---

## Part 2 — Distribution Preservation (KDE + KS-Test)

We check if each strategy keeps the original shape of the data intact.


In [ ]:
ks_results = []
for col in numerical_cols:
    orig = df_original[col].dropna().values
    for name, df in [('S1', df_s1), ('S2', df_s2), ('S3', df_s3)]:
        stat, pval = ks_2samp(orig, df[col].dropna().values)
        ks_results.append({'Column': col, 'Strategy': name, 'KS Stat': round(stat, 4), 'p-value': round(pval, 4)})

ks_df = pd.DataFrame(ks_results)
ks_df

In [ ]:
n = len(numerical_cols)
fig, axes = plt.subplots(n, 1, figsize=(12, 4 * n))
if n == 1:
    axes = [axes]
for ax, col in zip(axes, numerical_cols):
    sns.kdeplot(df_original[col].dropna(), ax=ax, label='Original', color='black', linewidth=2, linestyle='--')
    for name, df in [('S1', df_s1), ('S2', df_s2), ('S3', df_s3)]:
        sns.kdeplot(df[col].dropna(), ax=ax, label=name, color=colors[name], linewidth=1.5)
    ax.set_title(f'Distribution — {col}', fontsize=12, fontweight='bold')
    ax.legend()
plt.suptitle('Closer to Original (black dashed) = Better', fontsize=11, color='grey', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
cat_results = []
for col in categorical_cols:
    orig_dist = df_original[col].value_counts(normalize=True)
    for name, df in [('S1', df_s1), ('S2', df_s2), ('S3', df_s3)]:
        imp_dist = df[col].value_counts(normalize=True)
        all_cats = orig_dist.index.union(imp_dist.index)
        tvd      = 0.5 * sum(abs(orig_dist.get(c, 0) - imp_dist.get(c, 0)) for c in all_cats)
        cat_results.append({'Column': col, 'Strategy': name, 'TVD': round(tvd, 4)})

cat_df = pd.DataFrame(cat_results)
print('Total Variation Distance — Categorical (lower = better)')
cat_df

In [ ]:
summary_ks  = ks_df.groupby('Strategy')['KS Stat'].mean().round(4).to_frame()
summary_ks['Rank']  = summary_ks['KS Stat'].rank().astype(int)

summary_cat = cat_df.groupby('Strategy')['TVD'].mean().round(4).to_frame()
summary_cat['Rank'] = summary_cat['TVD'].rank().astype(int)

print('Numerical (KS Stat):')
display(summary_ks)
print('Categorical (TVD):')
display(summary_cat)

---

## Part 3 — Feature Correlation Preservation

How much does each strategy distort the correlation structure?


In [ ]:
corr_original    = df_original[numerical_cols].corr()
frobenius_scores = {}

for name, df in [('S1', df_s1), ('S2', df_s2), ('S3', df_s3)]:
    diff = corr_original - df[numerical_cols].corr()
    frobenius_scores[name] = round(np.linalg.norm(diff, 'fro'), 4)

frob_df = pd.DataFrame.from_dict(frobenius_scores, orient='index', columns=['Frobenius Norm'])
frob_df['Rank'] = frob_df['Frobenius Norm'].rank().astype(int)
frob_df

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, (title, df) in zip(axes, [('Original', df_original), ('S1', df_s1), ('S2', df_s2), ('S3', df_s3)]):
    sns.heatmap(df[numerical_cols].corr(), ax=ax, annot=True, fmt='.2f', cmap='coolwarm',
                vmin=-1, vmax=1, linewidths=0.5, cbar=False, annot_kws={'size': 7})
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)
plt.suptitle('Correlation Heatmaps — Closer to Original = Better', fontsize=12, color='grey', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(frobenius_scores.keys(), frobenius_scores.values(),
              color=[colors['S1'], colors['S2'], colors['S3']], edgecolor='white', width=0.5)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=10)
ax.set_title('Frobenius Norm — Correlation Distortion', fontsize=13, fontweight='bold')
ax.set_ylabel('Lower = Less Distortion')
plt.tight_layout()
plt.show()

# final


In [ ]:
final = pd.DataFrame(index=['S1', 'S2', 'S3'])
final.index.name = 'Strategy'

final['Masked Val (RMSE)']  = summary_masked['Rank (RMSE)'].values
final['Distribution (KS)']  = summary_ks['Rank'].values
final['Categorical (TVD)']  = summary_cat['Rank'].values
final['Correlation (Frob)'] = frob_df['Rank'].values
final['Total Score']        = final.sum(axis=1)
final['Overall Rank']       = final['Total Score'].rank().astype(int)

final.style.highlight_min(subset=['Total Score'], color='#c6efce')

# conclusion :

- using phylogeny distance and filling null/missing values with closest species weighted mean is best choice
